In [ ]:
session.sql("DROP TABLE IF EXISTS tmp_poi_stream_snapshot").collect()
print("Temp table dropped")

In [ ]:
session.sql("""
CREATE TEMP TABLE tmp_poi_stream_snapshot AS
SELECT *
FROM STREAM_T_PERSON_ORG_INVOLVEMENT
WHERE METADATA$ACTION = 'INSERT'
""").collect()

poi_raw = session.table("tmp_poi_stream_snapshot")
poi_raw = poi_raw.drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID")
print(f"snapshot person_org_involvement records found")

In [ ]:
ind_columns = ["SEARCHED_IND"]

poi_clean = poi_raw
for c in ind_columns:
    poi_clean = poi_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("N")
        ).otherwise(trim(col(c)))
    )

code_columns = ["REASON_CLOSED_CODE"]

for c in code_columns:
    poi_clean = poi_clean.with_column(
        c,
        upper(trim(col(c)))
    )

user_columns = ["ADD_USER", "MOD_USER"]

for c in user_columns:
    poi_clean = poi_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("SYSTEM")
        ).otherwise(trim(col(c)))
    )

trim_only_columns = ["SOURCE_FILE_NAME"]

for c in trim_only_columns:
    poi_clean = poi_clean.with_column(
        c,
        trim(col(c))
    )

poi_clean = poi_clean.with_column_renamed("LOAD_TIMESTAMP", "RAW_LOAD_TIMESTAMP")
print("poi clean")

In [ ]:
valid_poi = poi_clean.filter(
    col("POI_ID").is_not_null()
)

invalid_poi = poi_clean.filter(
    col("POI_ID").is_null()
).with_column(
    "ERROR_REASON",
    lit("POI_ID_NULL")
)
print("seperated valid and invalid records")

In [ ]:
checksum_columns = [
    ("POI_ID", "number"),
    ("SEARCHED_IND", "string"),
    ("REASON_CLOSED_CODE", "string"),
    ("ADD_USER", "string"),
    ("MOD_USER", "string"),
    ("INV_INV_ID", "number"),
    ("ADR_ADR_CALLED_FROM_ID", "number"),
    ("ORGN_ORGN_ID", "number"),
    ("PERSON_PERSON_ID", "number"),
    ("CAS_CAS_ID", "number"),
    ("IC_IC_ID", "number"),
    ("COMM_COMM_ID", "number"),
    ("TICKLER_ID", "number"),
    ("START_DATE", "timestamp"),
    ("END_DATE", "timestamp"),
]

checksum_exprs = []
for col_name, col_type in checksum_columns:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

poi_final = valid_poi.with_column(
    "CHECKSUM",
    sha2(concat_ws(lit("|"), *checksum_exprs), 256)
)

In [ ]:
poi_final = poi_final.with_column(
    "STAGING_LOADED_TIMESTAMP",
    current_timestamp()
)

poi_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_PERSON_ORG_INVOLVEMENT}",
    mode="truncate"
)

print(f"POI loaded into {STG}.{STG_PERSON_ORG_INVOLVEMENT}")

In [ ]:
invalid_poi.create_or_replace_temp_view("tmp_invalid_poi")

invalid_count = invalid_poi.count()

if invalid_count > 0:
    session.sql(f"""
    INSERT INTO {DB}.{STG}.{INVALID_RECORDS}
    (
        TABLE_NAME,
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        RAW_RECORD
    )
    SELECT
        'T_PERSON_ORG_INVOLVEMENT',
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        OBJECT_CONSTRUCT(*)
    FROM tmp_invalid_poi
    """).collect()
    print(f"Invalid records saved into {STG}.{INVALID_RECORDS}")
else:
    print("No invalid records")

In [ ]:
rows_processed = poi_final.count()
rows_failed = invalid_count

status = 'SUCCESS' if rows_failed == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    session.sql("SELECT UUID_STRING()").collect()[0][0],
    'STG_PERSON_ORG_INVOLVEMENT_LOAD',
    'STAGING',
    datetime.now(),
    datetime.now(),
    rows_processed,
    rows_failed,
    status,
    None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status,
    'STG_PERSON_ORG_INVOLVEMENT_LOAD',
    'STAGING',
    f'PERSON_ORG_INVOLVEMENT staging completed. Rows processed: {rows_processed}, Failed rows: {rows_failed}'
)
print("Audit log inserted and email sent")